# 04 Race Setup 2025 - Clean Race Laps

Scalony notebook dla wariantu wyscigowego 2025: audyt okrazen, definicja czystych okrazen wyscigowych, telemetria, cechy i modele.


## Source: `29_fetch_audit_2025_race_laps.py`


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import time
import traceback

import fastf1
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
EXPORT_DIR = BASE_DIR / "exports"
CACHE_DIR = BASE_DIR / "f1_cache"
RAW_DIR = EXPORT_DIR / "race_2025_raw_laps"

YEAR = 2025
SESSION_NAME = "R"
RATE_LIMIT_SLEEP_SECONDS = 3600

RAW_ALL_PATH = EXPORT_DIR / "all_race_laps_raw_2025.csv"
PROGRESS_PATH = EXPORT_DIR / "race_2025_fetch_progress.csv"
AUDIT_PATH = EXPORT_DIR / "race_2025_eligibility_audit.csv"
FILTER_SUMMARY_PATH = EXPORT_DIR / "race_2025_filter_summary.csv"
DRIVER_COUNTS_PATH = EXPORT_DIR / "race_2025_driver_clean_counts.csv"
EVENT_COUNTS_PATH = EXPORT_DIR / "race_2025_event_clean_counts.csv"
SESSION_SUMMARY_PATH = EXPORT_DIR / "race_2025_session_summary.csv"
LOG_PATH = EXPORT_DIR / "race_2025_fetch_audit.log"

SLICK_COMPOUNDS = {
    "SOFT", "MEDIUM", "HARD",
    "SUPERSOFT", "ULTRASOFT", "HYPERSOFT", "SUPERHARD",
    "C1", "C2", "C3", "C4", "C5",
    "XSOFT", "PRIME", "OPTION",
}
WET_COMPOUNDS = {"INTERMEDIATE", "WET"}


def log(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    try:
        print(line, flush=True)
    except UnicodeEncodeError:
        print(line.encode("ascii", errors="replace").decode("ascii"), flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def timed_to_seconds(series: pd.Series) -> pd.Series:
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()


def is_rate_limit_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(token in text for token in ["429", "rate limit", "ratelimit", "too many requests"])


def call_with_rate_limit_retry(fn, description: str):
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001
            if is_rate_limit_error(exc):
                log(
                    f"{description}: detected rate limit on attempt {attempt}. "
                    f"Sleeping for {RATE_LIMIT_SLEEP_SECONDS // 3600} hour before retry."
                )
                time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                continue
            raise


def build_schedule() -> pd.DataFrame:
    schedule = call_with_rate_limit_retry(
        lambda: fastf1.get_event_schedule(YEAR, include_testing=False),
        f"get_event_schedule({YEAR})",
    )
    schedule = schedule.copy()
    schedule = schedule[schedule["RoundNumber"].fillna(0).astype(int) > 0].copy()
    return schedule


def load_progress() -> pd.DataFrame:
    if PROGRESS_PATH.exists():
        return pd.read_csv(PROGRESS_PATH)
    return pd.DataFrame(columns=["season", "round", "event_name", "status", "n_laps_rows", "updated_at", "error_type", "error_message"])


def save_progress(df: pd.DataFrame) -> None:
    df.sort_values(["season", "round"], inplace=True)
    df.to_csv(PROGRESS_PATH, index=False)


def upsert_progress_row(progress_df: pd.DataFrame, row: dict) -> pd.DataFrame:
    mask = (progress_df["season"] == row["season"]) & (progress_df["round"] == row["round"])
    progress_df = progress_df.loc[~mask].copy()
    progress_df = pd.concat([progress_df, pd.DataFrame([row])], ignore_index=True)
    return progress_df


def fetch_race_session(year: int, round_number: int, event_name: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    session = fastf1.get_session(year, round_number, SESSION_NAME)
    call_with_rate_limit_retry(
        lambda: session.load(laps=True, telemetry=False, weather=True, messages=False),
        f"session.load({year}, R{round_number:02d}, {SESSION_NAME})",
    )
    laps = session.laps.copy()
    results = session.results.copy() if session.results is not None else pd.DataFrame()

    wanted_cols = [
        "Time", "Driver", "DriverNumber", "LapTime", "LapNumber", "Stint", "PitOutTime", "PitInTime",
        "Sector1Time", "Sector2Time", "Sector3Time",
        "Sector1SessionTime", "Sector2SessionTime", "Sector3SessionTime",
        "SpeedI1", "SpeedI2", "SpeedFL", "SpeedST",
        "IsPersonalBest", "Compound", "TyreLife", "FreshTyre",
        "Team", "TrackStatus", "Position", "Deleted", "DeletedReason",
        "FastF1Generated", "IsAccurate"
    ]
    existing_cols = [c for c in wanted_cols if c in laps.columns]
    laps = laps[existing_cols].copy()

    laps["season"] = year
    laps["round"] = round_number
    laps["event_name"] = event_name
    laps["session_name"] = SESSION_NAME
    laps["LapTimeSeconds"] = timed_to_seconds(laps["LapTime"]) if "LapTime" in laps.columns else np.nan
    laps["Sector1TimeSeconds"] = timed_to_seconds(laps["Sector1Time"]) if "Sector1Time" in laps.columns else np.nan
    laps["Sector2TimeSeconds"] = timed_to_seconds(laps["Sector2Time"]) if "Sector2Time" in laps.columns else np.nan
    laps["Sector3TimeSeconds"] = timed_to_seconds(laps["Sector3Time"]) if "Sector3Time" in laps.columns else np.nan

    weather_df = session.weather_data.copy() if getattr(session, "weather_data", None) is not None else pd.DataFrame()
    if not weather_df.empty and "Rainfall" in weather_df.columns:
        session_max_rainfall = weather_df["Rainfall"].max()
        session_is_dry_by_weather = not bool(session_max_rainfall)
    else:
        session_max_rainfall = np.nan
        session_is_dry_by_weather = np.nan
    laps["session_max_rainfall"] = session_max_rainfall
    laps["session_is_dry_by_weather"] = session_is_dry_by_weather

    if not results.empty:
        results["season"] = year
        results["round"] = round_number
        results["event_name"] = event_name
        results["session_name"] = SESSION_NAME

    return laps, results


def load_all_raw() -> pd.DataFrame:
    frames = []
    for path in sorted(RAW_DIR.glob("*_R_laps.csv")):
        try:
            df = pd.read_csv(path, low_memory=False)
        except Exception:
            continue
        if not df.empty:
            frames.append(df)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


def summarize_step(df: pd.DataFrame, step: str) -> dict:
    return {
        "step": step,
        "n_rows": len(df),
        "n_unique_drivers": df["Driver"].nunique() if "Driver" in df.columns else 0,
        "n_unique_sessions": df["session_key"].nunique() if "session_key" in df.columns else 0,
    }


def main() -> None:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")

    CACHE_DIR.mkdir(exist_ok=True)
    EXPORT_DIR.mkdir(exist_ok=True)
    RAW_DIR.mkdir(exist_ok=True)
    fastf1.Cache.enable_cache(str(CACHE_DIR))

    progress_df = load_progress()
    done_rounds = set(progress_df.loc[progress_df["status"] == "ok", "round"].astype(int).tolist())

    schedule_df = build_schedule()
    for _, row in schedule_df.iterrows():
        round_number = int(row["RoundNumber"])
        event_name = row["EventName"]
        if round_number in done_rounds:
            log(f"Skipping already fetched 2025 R{round_number:02d} {event_name}")
            continue

        log(f"Fetching 2025 R{round_number:02d} {event_name}")
        try:
            laps, results = fetch_race_session(YEAR, round_number, event_name)
            laps.to_csv(RAW_DIR / f"{YEAR}_R{round_number:02d}_R_laps.csv", index=False)
            results.to_csv(RAW_DIR / f"{YEAR}_R{round_number:02d}_R_results.csv", index=False)
            progress_df = upsert_progress_row(
                progress_df,
                {
                    "season": YEAR,
                    "round": round_number,
                    "event_name": event_name,
                    "status": "ok",
                    "n_laps_rows": len(laps),
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                    "error_type": None,
                    "error_message": None,
                },
            )
            save_progress(progress_df)
            log(f"Fetched 2025 R{round_number:02d} {event_name}: {len(laps)} laps")
        except Exception as exc:  # noqa: BLE001
            log(f"Failed 2025 R{round_number:02d} {event_name}: {type(exc).__name__}: {exc}")
            log(traceback.format_exc(limit=3).strip())
            progress_df = upsert_progress_row(
                progress_df,
                {
                    "season": YEAR,
                    "round": round_number,
                    "event_name": event_name,
                    "status": "failed",
                    "n_laps_rows": np.nan,
                    "updated_at": datetime.now().isoformat(timespec="seconds"),
                    "error_type": type(exc).__name__,
                    "error_message": str(exc),
                },
            )
            save_progress(progress_df)

    raw_df = load_all_raw()
    if raw_df.empty:
        log("No race raw laps downloaded; audit skipped.")
        return

    raw_df["session_key"] = raw_df["season"].astype(str) + "_" + raw_df["round"].astype(str) + "_" + raw_df["session_name"].astype(str)
    raw_df["TrackStatus_num"] = pd.to_numeric(raw_df["TrackStatus"], errors="coerce")
    raw_df["compound_upper"] = raw_df["Compound"].astype(str).str.upper().str.strip()
    raw_df["is_wet_compound"] = raw_df["compound_upper"].isin(WET_COMPOUNDS)
    raw_df["is_slick_compound"] = raw_df["compound_upper"].isin(SLICK_COMPOUNDS)
    raw_df["lap_time_not_null"] = raw_df["LapTimeSeconds"].notna()
    raw_df["dry_session"] = raw_df["session_is_dry_by_weather"] == True
    raw_df["accurate"] = raw_df["IsAccurate"].astype(str).str.lower().map({"true": True, "false": False}) == True
    raw_df["not_deleted"] = ~(
        raw_df["Deleted"].astype(str).str.lower().map({"true": True, "false": False}) == True
    )
    raw_df["not_fastf1_generated"] = ~(
        raw_df["FastF1Generated"].astype(str).str.lower().map({"true": True, "false": False}) == True
    )
    raw_df["track_status_green"] = raw_df["TrackStatus_num"] == 1
    raw_df["not_pit_in_out"] = raw_df["PitInTime"].isna() & raw_df["PitOutTime"].isna()

    step_00 = raw_df.copy()
    step_01 = step_00[step_00["lap_time_not_null"]].copy()
    step_02 = step_01[step_01["dry_session"]].copy()
    step_03 = step_02[step_02["accurate"]].copy()
    step_04 = step_03[step_03["not_deleted"]].copy()
    step_05 = step_04[step_04["not_fastf1_generated"]].copy()
    step_06 = step_05[step_05["track_status_green"]].copy()
    step_07 = step_06[step_06["not_pit_in_out"]].copy()

    # Stint-local outlier filter for "clean race pace" candidates
    step_07["stint_key"] = (
        step_07["season"].astype(str) + "_" +
        step_07["round"].astype(str) + "_" +
        step_07["Driver"].astype(str) + "_" +
        step_07["Stint"].astype(str)
    )
    stint_stats = (
        step_07.groupby("stint_key", dropna=False)["LapTimeSeconds"]
        .agg(stint_median="median")
        .reset_index()
    )
    step_07 = step_07.merge(stint_stats, on="stint_key", how="left")
    step_07["lap_delta_to_stint_median"] = step_07["LapTimeSeconds"] - step_07["stint_median"]
    step_07["within_2s_of_stint_median"] = step_07["lap_delta_to_stint_median"].abs() <= 2.0
    step_07["within_3s_of_stint_median"] = step_07["lap_delta_to_stint_median"].abs() <= 3.0

    step_08 = step_07[step_07["within_2s_of_stint_median"]].copy()

    raw_df.to_csv(RAW_ALL_PATH, index=False)
    audit_cols = [
        "season", "round", "event_name", "session_name", "Driver", "Team", "LapNumber", "Stint",
        "LapTimeSeconds", "Compound", "TyreLife", "TrackStatus",
        "lap_time_not_null", "dry_session", "accurate", "not_deleted",
        "not_fastf1_generated", "track_status_green", "not_pit_in_out",
        "lap_delta_to_stint_median", "within_2s_of_stint_median", "within_3s_of_stint_median"
    ]
    step_07[audit_cols].to_csv(AUDIT_PATH, index=False)

    filter_rows = [
        summarize_step(step_00, "00_raw_race_laps"),
        summarize_step(step_01, "01_lap_time_not_null"),
        summarize_step(step_02, "02_dry_sessions_only"),
        summarize_step(step_03, "03_is_accurate_true"),
        summarize_step(step_04, "04_not_deleted"),
        summarize_step(step_05, "05_not_fastf1_generated"),
        summarize_step(step_06, "06_track_status_eq_1"),
        summarize_step(step_07, "07_not_pit_in_out"),
        summarize_step(step_08, "08_within_2s_of_stint_median"),
    ]
    pd.DataFrame(filter_rows).to_csv(FILTER_SUMMARY_PATH, index=False)

    driver_counts_df = (
        step_08.groupby("Driver", dropna=False)
        .agg(
            n_clean_race_laps=("LapNumber", "count"),
            n_sessions=("session_key", "nunique"),
            n_events=("round", "nunique"),
        )
        .reset_index()
        .sort_values(["n_clean_race_laps", "Driver"], ascending=[False, True])
    )
    driver_counts_df.to_csv(DRIVER_COUNTS_PATH, index=False)

    event_counts_df = (
        step_08.groupby(["round", "event_name"], dropna=False)
        .agg(
            n_clean_race_laps=("LapNumber", "count"),
            n_drivers=("Driver", "nunique"),
        )
        .reset_index()
        .sort_values("round")
    )
    event_counts_df.to_csv(EVENT_COUNTS_PATH, index=False)

    session_summary_df = (
        raw_df.groupby(["round", "event_name"], dropna=False)
        .agg(
            n_raw_laps=("LapNumber", "count"),
            n_valid_green_non_pit_laps=("track_status_green", lambda s: int(s.sum())),
            is_dry_session=("dry_session", lambda s: bool(s.iloc[0]) if len(s) else False),
        )
        .reset_index()
        .sort_values("round")
    )
    session_summary_df.to_csv(SESSION_SUMMARY_PATH, index=False)

    log("2025 race fetch and eligibility audit complete")


if __name__ == "__main__":
    main()


## Source: `30_prepare_2025_race_top5_targets.py`


In [ ]:
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("exports")
AUDIT_PATH = EXPORT_DIR / "race_2025_eligibility_audit.csv"
TARGET_DRIVERS = ["SAI", "VER", "LEC", "OCO", "HAM"]


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def main() -> None:
    audit_df = pd.read_csv(AUDIT_PATH)
    audit_df["session_key"] = (
        audit_df["season"].astype(str) + "_" +
        audit_df["round"].astype(str) + "_" +
        audit_df["session_name"].astype(str)
    )
    audit_df["lap_key"] = (
        audit_df["season"].astype(int).astype(str) + "_r" +
        audit_df["round"].astype(int).astype(str).str.zfill(2) + "_" +
        audit_df["session_name"].astype(str) + "_" +
        audit_df["Driver"].astype(str) + "_lap" +
        audit_df["LapNumber"].astype(int).astype(str)
    )

    target_df = audit_df[
        audit_df["Driver"].isin(TARGET_DRIVERS) &
        (audit_df["within_2s_of_stint_median"] == True)
    ].copy()
    target_df = target_df.sort_values(["round", "Driver", "LapNumber"]).reset_index(drop=True)

    export_csv(target_df, "race_2025_balanced_top5_clean_targets")

    driver_counts_df = (
        target_df.groupby("Driver", dropna=False)
        .agg(
            n_laps=("lap_key", "count"),
            n_events=("round", "nunique"),
            n_sessions=("session_key", "nunique"),
        )
        .reset_index()
        .sort_values(["n_laps", "Driver"], ascending=[False, True])
    )
    export_csv(driver_counts_df, "race_2025_balanced_top5_clean_target_counts")

    event_counts_df = (
        target_df.groupby(["round", "event_name"], dropna=False)
        .agg(
            n_laps=("lap_key", "count"),
            n_drivers=("Driver", "nunique"),
        )
        .reset_index()
        .sort_values("round")
    )
    export_csv(event_counts_df, "race_2025_balanced_top5_clean_target_event_counts")

    print("Prepared 2025 race clean target list for balanced_top5 drivers.")
    print(driver_counts_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `31_extract_race_2025_top5_telemetry.py`


In [ ]:
from __future__ import annotations

from datetime import datetime
from pathlib import Path
import sys
import time
import traceback

import fastf1
import numpy as np
import pandas as pd


BASE_DIR = Path(".")
EXPORT_DIR = BASE_DIR / "exports"
CACHE_DIR = BASE_DIR / "f1_cache"
TARGET_PATH = EXPORT_DIR / "race_2025_balanced_top5_clean_targets.csv"

AUDIT_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_audit.csv"
MERGED_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_merged.csv"
SUCCESS_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_success_summary.csv"
DRIVER_COUNTS_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_driver_counts.csv"
LOG_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_extract.log"

RATE_LIMIT_SLEEP_SECONDS = 3600


def log(message: str) -> None:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {message}"
    try:
        print(line, flush=True)
    except UnicodeEncodeError:
        safe_line = line.encode("ascii", errors="replace").decode("ascii")
        print(safe_line, flush=True)
    with LOG_PATH.open("a", encoding="utf-8") as handle:
        handle.write(line + "\n")


def timed_to_seconds(series: pd.Series) -> pd.Series:
    return pd.to_timedelta(series, errors="coerce").dt.total_seconds()


def is_rate_limit_error(exc: Exception) -> bool:
    text = f"{type(exc).__name__}: {exc}".lower()
    return any(token in text for token in ["429", "rate limit", "ratelimit", "too many requests"])


def call_with_rate_limit_retry(fn, description: str):
    attempt = 0
    while True:
        attempt += 1
        try:
            return fn()
        except Exception as exc:  # noqa: BLE001
            if is_rate_limit_error(exc):
                log(
                    f"{description}: detected rate limit on attempt {attempt}. "
                    f"Sleeping for {RATE_LIMIT_SLEEP_SECONDS // 3600} hour before retry."
                )
                time.sleep(RATE_LIMIT_SLEEP_SECONDS)
                continue
            raise


def load_existing_done_lap_keys() -> set[str]:
    if not AUDIT_PATH.exists():
        return set()
    audit_df = pd.read_csv(AUDIT_PATH)
    return set(audit_df.loc[audit_df["status"] == "ok", "lap_key"].astype(str))


def append_csv_rows(path: Path, rows: list[dict]) -> None:
    if not rows:
        return
    df = pd.DataFrame(rows)
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def append_df(path: Path, df: pd.DataFrame) -> None:
    if df.empty:
        return
    header = not path.exists()
    df.to_csv(path, mode="a", header=header, index=False)


def main() -> None:
    if hasattr(sys.stdout, "reconfigure"):
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    if hasattr(sys.stderr, "reconfigure"):
        sys.stderr.reconfigure(encoding="utf-8", errors="replace")

    fastf1.Cache.enable_cache(str(CACHE_DIR))

    targets_df = pd.read_csv(TARGET_PATH)
    if "lap_key" not in targets_df.columns:
        raise ValueError("Target file must include lap_key column.")

    done_lap_keys = load_existing_done_lap_keys()
    pending_df = targets_df[~targets_df["lap_key"].astype(str).isin(done_lap_keys)].copy()

    log(
        f"Race telemetry extraction target laps: {len(targets_df)}; "
        f"already done: {len(done_lap_keys)}; pending: {len(pending_df)}"
    )
    if pending_df.empty:
        log("No pending laps. Building summaries only.")
    else:
        grouped = pending_df.groupby(["season", "round"], sort=True)

        for (season, round_number), group in grouped:
            log(f"Loading session {season} R{int(round_number):02d} R with {len(group)} pending laps")
            telemetry_rows: list[pd.DataFrame] = []
            audit_rows: list[dict] = []

            try:
                session = fastf1.get_session(int(season), int(round_number), "R")
                call_with_rate_limit_retry(
                    lambda: session.load(laps=True, telemetry=True, weather=False, messages=False),
                    f"session.load({season}, R{int(round_number):02d}, R, telemetry=True)",
                )
                laps = session.laps.copy()

                for _, target in group.iterrows():
                    driver = target["Driver"]
                    lap_number = target["LapNumber"]
                    lap_key = str(target["lap_key"])

                    try:
                        candidate = laps[
                            (laps["Driver"] == driver) &
                            (laps["LapNumber"] == lap_number)
                        ].copy()

                        if len(candidate) != 1:
                            audit_rows.append(
                                {
                                    "lap_key": lap_key,
                                    "season": season,
                                    "round": round_number,
                                    "Driver": driver,
                                    "LapNumber": lap_number,
                                    "status": "lap_match_not_unique",
                                    "n_matches": len(candidate),
                                }
                            )
                            continue

                        lap = candidate.iloc[0]
                        car_data = lap.get_car_data().copy()
                        pos_data = lap.get_pos_data().copy()

                        if "Time" in car_data.columns:
                            car_data["telemetry_time_sec"] = timed_to_seconds(car_data["Time"])
                        if "SessionTime" in car_data.columns:
                            car_data["telemetry_session_time_sec"] = timed_to_seconds(car_data["SessionTime"])
                        if "Date" in car_data.columns:
                            car_data["telemetry_date"] = car_data["Date"].astype(str)

                        if "Time" in pos_data.columns:
                            pos_data["pos_time_sec"] = timed_to_seconds(pos_data["Time"])
                        if "SessionTime" in pos_data.columns:
                            pos_data["pos_session_time_sec"] = timed_to_seconds(pos_data["SessionTime"])
                        if "Date" in pos_data.columns:
                            pos_data["pos_date"] = pos_data["Date"].astype(str)

                        for df_ in [car_data, pos_data]:
                            df_["lap_key"] = lap_key
                            df_["season"] = season
                            df_["round"] = round_number
                            df_["event_name"] = target.get("event_name")
                            df_["session_name"] = target.get("session_name")
                            df_["Driver"] = driver
                            df_["LapNumber"] = lap_number

                        for col in [
                            "LapTimeSeconds",
                            "Compound",
                            "TyreLife",
                            "Team",
                            "Stint",
                            "SessionPart",
                            "stint_median_lap_seconds",
                            "lap_time_delta_to_stint_median",
                            "session_key",
                        ]:
                            car_data[col] = target.get(col)

                        pos_keep = [
                            c for c in pos_data.columns if c in [
                                "lap_key", "X", "Y", "Z", "Status",
                                "pos_time_sec", "pos_session_time_sec", "pos_date"
                            ]
                        ]
                        pos_data = pos_data[pos_keep].copy()

                        car_data = car_data.reset_index(drop=True)
                        pos_data = pos_data.reset_index(drop=True)
                        n = min(len(car_data), len(pos_data))

                        merged = pd.concat(
                            [
                                car_data.iloc[:n].reset_index(drop=True),
                                pos_data.drop(columns=["lap_key"], errors="ignore").iloc[:n].reset_index(drop=True),
                            ],
                            axis=1,
                        )
                        merged["sample_idx"] = np.arange(len(merged))
                        telemetry_rows.append(merged)

                        audit_rows.append(
                            {
                                "lap_key": lap_key,
                                "season": season,
                                "round": round_number,
                                "event_name": target.get("event_name"),
                                "Driver": driver,
                                "LapNumber": lap_number,
                                "status": "ok",
                                "car_rows": len(car_data),
                                "pos_rows": len(pos_data),
                                "merged_rows": len(merged),
                                "Compound": target.get("Compound"),
                                "TyreLife": target.get("TyreLife"),
                                "lap_time_delta_to_stint_median": target.get("lap_time_delta_to_stint_median"),
                            }
                        )
                    except Exception as exc:  # noqa: BLE001
                        audit_rows.append(
                            {
                                "lap_key": lap_key,
                                "season": season,
                                "round": round_number,
                                "event_name": target.get("event_name"),
                                "Driver": driver,
                                "LapNumber": lap_number,
                                "status": "lap_extract_error",
                                "error_type": type(exc).__name__,
                                "error_msg": str(exc),
                            }
                        )

                append_df(MERGED_PATH, pd.concat(telemetry_rows, ignore_index=True) if telemetry_rows else pd.DataFrame())
                append_csv_rows(AUDIT_PATH, audit_rows)
                ok_count = sum(1 for row in audit_rows if row["status"] == "ok")
                log(f"Completed {season} R{int(round_number):02d} R: {ok_count}/{len(group)} laps ok")

            except Exception as exc:  # noqa: BLE001
                log(f"Session failed {season} R{int(round_number):02d} R: {type(exc).__name__}: {exc}")
                log(traceback.format_exc(limit=3).strip())
                failure_rows = [
                    {
                        "lap_key": str(target["lap_key"]),
                        "season": season,
                        "round": round_number,
                        "event_name": target.get("event_name"),
                        "Driver": target["Driver"],
                        "LapNumber": target["LapNumber"],
                        "status": "session_load_error",
                        "error_type": type(exc).__name__,
                        "error_msg": str(exc),
                    }
                    for _, target in group.iterrows()
                ]
                append_csv_rows(AUDIT_PATH, failure_rows)

    if AUDIT_PATH.exists():
        audit_df = pd.read_csv(AUDIT_PATH)
        success_summary_df = (
            audit_df.groupby("status", dropna=False)
            .size()
            .reset_index(name="count")
            .sort_values("count", ascending=False)
        )
        success_summary_df["fraction"] = success_summary_df["count"] / len(audit_df)
        success_summary_df.to_csv(SUCCESS_PATH, index=False)

        ok_keys = set(audit_df.loc[audit_df["status"] == "ok", "lap_key"].astype(str))
        completed_targets = targets_df[targets_df["lap_key"].astype(str).isin(ok_keys)].copy()
        driver_counts_df = (
            completed_targets.groupby("Driver", dropna=False)
            .agg(n_laps=("lap_key", "count"), n_events=("round", "nunique"))
            .reset_index()
            .sort_values(["n_laps", "Driver"], ascending=[False, True])
        )
        driver_counts_df.to_csv(DRIVER_COUNTS_PATH, index=False)

    log("Race telemetry extraction script finished")


if __name__ == "__main__":
    main()


## Source: `32_summarize_race_2025_telemetry.py`


In [ ]:
from pathlib import Path

import pandas as pd


EXPORT_DIR = Path("exports")
AUDIT_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_audit.csv"
MERGED_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_merged.csv"
TARGETS_PATH = EXPORT_DIR / "race_2025_balanced_top5_clean_targets.csv"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def main() -> None:
    audit_df = pd.read_csv(AUDIT_PATH)
    targets_df = pd.read_csv(TARGETS_PATH)
    merged_df = pd.read_csv(MERGED_PATH, low_memory=False)

    ok_audit_df = audit_df[audit_df["status"] == "ok"].copy()

    export_csv(ok_audit_df.sort_values(["round", "Driver", "LapNumber"]), "race_2025_balanced_top5_telemetry_sequence_lengths")

    summary_df = pd.DataFrame(
        {
            "target_laps": [len(targets_df)],
            "extracted_ok_laps": [len(ok_audit_df)],
            "extraction_success_rate": [len(ok_audit_df) / len(targets_df) if len(targets_df) else None],
            "merged_rows_total": [len(merged_df)],
            "merged_rows_mean": [ok_audit_df["merged_rows"].mean()],
            "merged_rows_median": [ok_audit_df["merged_rows"].median()],
            "merged_rows_min": [ok_audit_df["merged_rows"].min()],
            "merged_rows_max": [ok_audit_df["merged_rows"].max()],
            "events_covered": [ok_audit_df["round"].nunique()],
            "drivers_covered": [ok_audit_df["Driver"].nunique()],
        }
    )
    export_csv(summary_df, "race_2025_balanced_top5_telemetry_sequence_summary")

    driver_summary_df = (
        ok_audit_df.groupby("Driver", dropna=False)
        .agg(
            n_laps=("lap_key", "count"),
            n_events=("round", "nunique"),
            merged_rows_mean=("merged_rows", "mean"),
            merged_rows_median=("merged_rows", "median"),
            merged_rows_min=("merged_rows", "min"),
            merged_rows_max=("merged_rows", "max"),
        )
        .reset_index()
        .sort_values(["n_laps", "Driver"], ascending=[False, True])
    )
    export_csv(driver_summary_df, "race_2025_balanced_top5_telemetry_driver_sequence_summary")

    event_summary_df = (
        ok_audit_df.groupby(["round", "event_name"], dropna=False)
        .agg(
            n_laps=("lap_key", "count"),
            n_drivers=("Driver", "nunique"),
            merged_rows_mean=("merged_rows", "mean"),
            merged_rows_median=("merged_rows", "median"),
        )
        .reset_index()
        .sort_values("round")
    )
    export_csv(event_summary_df, "race_2025_balanced_top5_telemetry_event_sequence_summary")

    channel_rows = []
    candidate_channels = [
        "RPM", "Speed", "nGear", "Throttle", "Brake", "DRS",
        "X", "Y", "Z", "telemetry_time_sec", "telemetry_session_time_sec",
        "pos_time_sec", "pos_session_time_sec",
    ]
    for col in candidate_channels:
        if col not in merged_df.columns:
            channel_rows.append({"column": col, "present": False, "non_null_fraction": None})
            continue
        channel_rows.append(
            {
                "column": col,
                "present": True,
                "non_null_fraction": merged_df[col].notna().mean(),
            }
        )
    channel_df = pd.DataFrame(channel_rows)
    export_csv(channel_df, "race_2025_balanced_top5_telemetry_channel_availability")

    print("Race 2025 telemetry summary ready.")
    print(summary_df.to_string(index=False))
    print(driver_summary_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `33_build_race_2025_features.py`


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd


EXPORT_DIR = Path("exports")
TELEMETRY_PATH = EXPORT_DIR / "race_2025_balanced_top5_telemetry_merged.csv"
TARGETS_PATH = EXPORT_DIR / "race_2025_balanced_top5_clean_targets.csv"


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def safe_frac(mask: pd.Series) -> float:
    if len(mask) == 0:
        return np.nan
    return float(mask.mean())


def first_true_frac_pos(mask: pd.Series) -> float:
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        return np.nan
    denom = max(len(mask) - 1, 1)
    return float(idx[0] / denom)


def last_true_frac_pos(mask: pd.Series) -> float:
    idx = np.flatnonzero(mask.to_numpy())
    if len(idx) == 0:
        return np.nan
    denom = max(len(mask) - 1, 1)
    return float(idx[-1] / denom)


def count_state_changes(mask: pd.Series) -> int:
    values = mask.astype(int).to_numpy()
    if len(values) <= 1:
        return 0
    return int(np.abs(np.diff(values)).sum())


def build_features_for_lap(lap_df: pd.DataFrame) -> dict:
    lap_df = lap_df.sort_values("sample_idx").reset_index(drop=True)

    speed = pd.to_numeric(lap_df["Speed"], errors="coerce")
    throttle = pd.to_numeric(lap_df["Throttle"], errors="coerce")
    rpm = pd.to_numeric(lap_df["RPM"], errors="coerce")
    gear = pd.to_numeric(lap_df["nGear"], errors="coerce")
    brake = (
        lap_df["Brake"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({"true": 1.0, "false": 0.0})
    )
    drs = pd.to_numeric(lap_df["DRS"], errors="coerce") if "DRS" in lap_df.columns else pd.Series(np.nan, index=lap_df.index)

    throttle_zero = throttle <= 0.0
    throttle_full = throttle >= 99.0
    throttle_mid = (throttle > 0.0) & (throttle < 99.0)
    brake_active = brake >= 0.5

    speed_diff = speed.diff()
    throttle_diff = throttle.diff()
    rpm_diff = rpm.diff()
    gear_diff = gear.diff()

    features = {
        "n_samples": len(lap_df),
        "speed_mean": speed.mean(),
        "speed_std": speed.std(),
        "speed_min": speed.min(),
        "speed_max": speed.max(),
        "speed_q10": speed.quantile(0.10),
        "speed_q50": speed.quantile(0.50),
        "speed_q90": speed.quantile(0.90),
        "speed_range": speed.max() - speed.min(),
        "throttle_mean": throttle.mean(),
        "throttle_std": throttle.std(),
        "throttle_min": throttle.min(),
        "throttle_max": throttle.max(),
        "throttle_zero_frac": safe_frac(throttle_zero),
        "throttle_full_frac": safe_frac(throttle_full),
        "throttle_mid_frac": safe_frac(throttle_mid),
        "brake_mean": brake.mean(),
        "brake_std": brake.std(),
        "brake_min": brake.min(),
        "brake_max": brake.max(),
        "brake_active_frac": safe_frac(brake_active),
        "brake_on_count": count_state_changes(brake_active) // 2 + int(brake_active.iloc[0]) if len(brake_active) else 0,
        "first_brake_frac_pos": first_true_frac_pos(brake_active),
        "last_brake_frac_pos": last_true_frac_pos(brake_active),
        "rpm_mean": rpm.mean(),
        "rpm_std": rpm.std(),
        "rpm_min": rpm.min(),
        "rpm_max": rpm.max(),
        "gear_mean": gear.mean(),
        "gear_std": gear.std(),
        "gear_min": gear.min(),
        "gear_max": gear.max(),
        "gear_change_count": int(gear_diff.fillna(0).ne(0).sum()),
        "speed_diff_mean": speed_diff.mean(),
        "speed_diff_std": speed_diff.std(),
        "speed_diff_abs_mean": speed_diff.abs().mean(),
        "throttle_diff_mean": throttle_diff.mean(),
        "throttle_diff_std": throttle_diff.std(),
        "throttle_diff_abs_mean": throttle_diff.abs().mean(),
        "rpm_diff_mean": rpm_diff.mean(),
        "rpm_diff_std": rpm_diff.std(),
        "rpm_diff_abs_mean": rpm_diff.abs().mean(),
        "drs_active_frac": safe_frac(drs > 0),
    }
    return features


def main() -> None:
    telemetry_df = pd.read_csv(TELEMETRY_PATH, low_memory=False)
    targets_df = pd.read_csv(TARGETS_PATH)

    lap_features = []
    for lap_key, lap_df in telemetry_df.groupby("lap_key", sort=True):
        meta = lap_df.iloc[0]
        feature_row = build_features_for_lap(lap_df)
        feature_row.update(
            {
                "lap_key": lap_key,
                "Driver": meta["Driver"],
                "season": int(meta["season"]),
                "round": int(meta["round"]),
                "event_name": meta.get("event_name"),
                "session_name": meta.get("session_name"),
                "LapNumber": meta["LapNumber"],
                "LapTimeSeconds": meta.get("LapTimeSeconds"),
                "Compound": meta.get("Compound"),
                "TyreLife": meta.get("TyreLife"),
                "Team": meta.get("Team"),
                "Stint": meta.get("Stint"),
                "SessionPart": meta.get("SessionPart"),
                "stint_median_lap_seconds": meta.get("stint_median_lap_seconds"),
                "lap_time_delta_to_stint_median": meta.get("lap_time_delta_to_stint_median"),
                "session_key": meta.get("session_key"),
            }
        )
        lap_features.append(feature_row)

    features_df = pd.DataFrame(lap_features)
    features_df = features_df.merge(
        targets_df[
            [
                "lap_key",
                "TrackStatus",
                "lap_time_not_null",
                "dry_session",
                "accurate",
                "not_deleted",
                "not_fastf1_generated",
                "track_status_green",
                "not_pit_in_out",
                "within_2s_of_stint_median",
                "within_3s_of_stint_median",
            ]
        ],
        on="lap_key",
        how="left",
    )

    export_csv(features_df, "race_2025_balanced_top5_lap_features")

    missingness_df = (
        features_df.isna().mean().reset_index()
        .rename(columns={"index": "feature", 0: "missing_fraction"})
        .sort_values(["missing_fraction", "feature"], ascending=[False, True])
    )
    export_csv(missingness_df, "race_2025_balanced_top5_lap_features_missingness")

    summary_stats_df = features_df.describe(include="all").transpose().reset_index().rename(columns={"index": "feature"})
    export_csv(summary_stats_df, "race_2025_balanced_top5_lap_features_summary_stats")

    class_balance_df = (
        features_df.groupby("Driver", dropna=False)
        .size()
        .reset_index(name="n_laps")
        .sort_values(["n_laps", "Driver"], ascending=[False, True])
    )
    export_csv(class_balance_df, "race_2025_balanced_top5_lap_features_class_balance")

    print("Built 2025 race handcrafted feature table.")
    print(class_balance_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `34_race_2025_baseline.py`


In [ ]:
from pathlib import Path
import importlib.util

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler


EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def has_xgboost() -> bool:
    return importlib.util.find_spec("xgboost") is not None


def build_model(model_name: str, numeric_features: list[str], categorical_features: list[str]):
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    steps=[
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        ("onehot", OneHotEncoder(handle_unknown="ignore")),
                    ]
                ),
                categorical_features,
            ),
        ]
    )

    if model_name == "logistic_regression":
        estimator = LogisticRegression(
            penalty="l1",
            C=2.0,
            solver="saga",
            max_iter=5000,
            random_state=RANDOM_STATE,
        )
    elif model_name == "random_forest":
        estimator = RandomForestClassifier(
            n_estimators=400,
            max_depth=None,
            min_samples_leaf=1,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )
    elif model_name == "hist_gradient_boosting":
        estimator = HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_depth=6,
            max_iter=300,
            random_state=RANDOM_STATE,
        )
    elif model_name == "xgboost":
        from xgboost import XGBClassifier

        estimator = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="multi:softprob",
            eval_metric="mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )
    else:
        raise ValueError(model_name)

    return Pipeline(steps=[("preprocessor", preprocessor), ("model", estimator)])


def run_experiment(df: pd.DataFrame, experiment_name: str, model_names: list[str], feature_cols: list[str]):
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(df["Driver"])
    groups = df["session_key"].astype(str).to_numpy()

    X = df[feature_cols].copy()
    categorical_features = [c for c in feature_cols if X[c].dtype == "object"]
    numeric_features = [c for c in feature_cols if c not in categorical_features]

    outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

    fold_rows = []
    summary_rows = []

    for model_name in model_names:
        oof_pred = np.full(len(df), -1, dtype=int)

        for fold, (train_idx, test_idx) in enumerate(outer_cv.split(X, y, groups), start=1):
            x_train = X.iloc[train_idx]
            x_test = X.iloc[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]

            model = build_model(model_name, numeric_features, categorical_features)
            model.fit(x_train, y_train)

            train_pred = model.predict(x_train)
            test_pred = model.predict(x_test)
            oof_pred[test_idx] = test_pred

            fold_rows.append(
                {
                    "experiment": experiment_name,
                    "model": model_name,
                    "fold": fold,
                    "n_train": len(train_idx),
                    "n_test": len(test_idx),
                    "train_accuracy": accuracy_score(y_train, train_pred),
                    "test_accuracy": accuracy_score(y_test, test_pred),
                    "train_balanced_accuracy": balanced_accuracy_score(y_train, train_pred),
                    "test_balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
                    "train_macro_f1": f1_score(y_train, train_pred, average="macro"),
                    "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
                }
            )

        summary_rows.append(
            {
                "experiment": experiment_name,
                "model": model_name,
                "n_samples": len(df),
                "n_drivers": df["Driver"].nunique(),
                "n_features": len(feature_cols),
                "n_sessions": df["session_key"].nunique(),
                "oof_accuracy": accuracy_score(y, oof_pred),
                "oof_balanced_accuracy": balanced_accuracy_score(y, oof_pred),
                "oof_macro_f1": f1_score(y, oof_pred, average="macro"),
            }
        )

    return pd.DataFrame(fold_rows), pd.DataFrame(summary_rows)


def main() -> None:
    df = pd.read_csv(EXPORT_DIR / "race_2025_balanced_top5_lap_features.csv")

    base_drop = [
        "lap_key",
        "Driver",
        "season",
        "round",
        "event_name",
        "session_name",
        "LapNumber",
        "Team",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
    ]
    weak_drop = [
        "throttle_min",
        "brake_min",
    ]
    time_drop = [
        "LapTimeSeconds",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
    ]
    race_context_drop = [
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
        "drs_active_frac",
    ]

    experiments = {
        "full_race_context": [c for c in df.columns if c not in base_drop + weak_drop],
        "without_time_features": [c for c in df.columns if c not in base_drop + weak_drop + time_drop],
        "style_core_no_time_no_race_context": [
            c for c in df.columns if c not in base_drop + weak_drop + time_drop + race_context_drop
        ],
    }
    all_missing_cols = [c for c in df.columns if df[c].isna().all()]
    experiments = {
        name: [c for c in feature_cols if c not in all_missing_cols]
        for name, feature_cols in experiments.items()
    }

    model_names = ["logistic_regression", "random_forest", "hist_gradient_boosting"]
    if has_xgboost():
        model_names.append("xgboost")

    all_fold_rows = []
    all_summary_rows = []

    for experiment_name, feature_cols in experiments.items():
        fold_df, summary_df = run_experiment(df, experiment_name, model_names, feature_cols)
        all_fold_rows.append(fold_df)
        all_summary_rows.append(summary_df)

    fold_metrics_df = pd.concat(all_fold_rows, ignore_index=True)
    summary_df = pd.concat(all_summary_rows, ignore_index=True)

    grouped_summary_df = (
        fold_metrics_df.groupby(["experiment", "model"], dropna=False)
        .agg(
            mean_train_accuracy=("train_accuracy", "mean"),
            mean_test_accuracy=("test_accuracy", "mean"),
            mean_train_macro_f1=("train_macro_f1", "mean"),
            mean_test_macro_f1=("test_macro_f1", "mean"),
        )
        .reset_index()
    )
    grouped_summary_df["mean_accuracy_gap"] = grouped_summary_df["mean_train_accuracy"] - grouped_summary_df["mean_test_accuracy"]
    grouped_summary_df["mean_macro_f1_gap"] = grouped_summary_df["mean_train_macro_f1"] - grouped_summary_df["mean_test_macro_f1"]

    summary_df = summary_df.merge(grouped_summary_df, on=["experiment", "model"], how="left")
    summary_df = summary_df.sort_values(["oof_macro_f1", "oof_accuracy"], ascending=[False, False]).reset_index(drop=True)

    best_models_df = summary_df.head(3).reset_index(drop=True)

    export_csv(fold_metrics_df, "race_2025_baseline_fold_metrics")
    export_csv(summary_df, "race_2025_baseline_model_summary")
    export_csv(best_models_df, "race_2025_baseline_best_models")
    export_csv(
        pd.DataFrame({"feature": all_missing_cols}).sort_values("feature").reset_index(drop=True),
        "race_2025_baseline_all_missing_features",
    )

    print("Race 2025 baseline modeling complete.")
    print(best_models_df.to_string(index=False))


if __name__ == "__main__":
    main()


## Source: `35_race_2025_sequence_models.py`


In [ ]:
from pathlib import Path
import os
import warnings

import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder


warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

EXPORT_DIR = Path("exports")
RANDOM_STATE = 42
N_SPLITS = 5
SEQ_LEN = 300
SEQ_CHANNELS = ["Speed", "Throttle", "Brake", "RPM", "nGear"]
MAX_EPOCHS = 60
BATCH_SIZE = 32
PATIENCE = 8

tf.keras.utils.set_random_seed(RANDOM_STATE)


def export_csv(df: pd.DataFrame, name: str) -> Path:
    path = EXPORT_DIR / f"{name}.csv"
    df.to_csv(path, index=False)
    return path


def resample_sequence(values: np.ndarray, target_len: int) -> np.ndarray:
    values = pd.Series(values).astype(float).interpolate(limit_direction="both").bfill().ffill()
    values = values.to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return np.zeros(target_len, dtype=float)
    if n == 1:
        return np.repeat(values[0], target_len)
    x_old = np.linspace(0.0, 1.0, n)
    x_new = np.linspace(0.0, 1.0, target_len)
    return np.interp(x_new, x_old, values)


def standardize_sequence_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.reshape(-1, train_x.shape[-1]).mean(axis=0)
    std = train_x.reshape(-1, train_x.shape[-1]).std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std


def standardize_by_train(train_x: np.ndarray, val_x: np.ndarray, test_x: np.ndarray):
    mean = train_x.mean(axis=0)
    std = train_x.std(axis=0)
    std = np.where(std > 0, std, 1.0)
    return (train_x - mean) / std, (val_x - mean) / std, (test_x - mean) / std


def build_cnn_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(inp)
    x = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x)
    x = tf.keras.layers.MaxPooling1D(2)(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_hybrid_cnn_model(seq_len: int, n_channels: int, n_tab_features: int, n_classes: int) -> tf.keras.Model:
    seq_in = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    tab_in = tf.keras.Input(shape=(n_tab_features,), name="tab_input")

    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(seq_in)
    x_seq = tf.keras.layers.Conv1D(32, 7, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.MaxPooling1D(2)(x_seq)
    x_seq = tf.keras.layers.Dropout(0.2)(x_seq)
    x_seq = tf.keras.layers.Conv1D(64, 5, padding="same", activation="relu")(x_seq)
    x_seq = tf.keras.layers.GlobalAveragePooling1D()(x_seq)

    x_tab = tf.keras.layers.Dense(32, activation="relu")(tab_in)
    x_tab = tf.keras.layers.Dropout(0.2)(x_tab)

    x = tf.keras.layers.Concatenate()([x_seq, x_tab])
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=[seq_in, tab_in], outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_gru_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.GRU(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.GRU(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_lstm_model(seq_len: int, n_channels: int, n_classes: int) -> tf.keras.Model:
    inp = tf.keras.Input(shape=(seq_len, n_channels), name="seq_input")
    x = tf.keras.layers.LSTM(64, return_sequences=True, dropout=0.2)(inp)
    x = tf.keras.layers.LSTM(32, dropout=0.2)(x)
    x = tf.keras.layers.Dense(64, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    out = tf.keras.layers.Dense(n_classes, activation="softmax")(x)
    model = tf.keras.Model(inputs=inp, outputs=out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def prepare_data():
    telemetry_df = pd.read_csv(EXPORT_DIR / "race_2025_balanced_top5_telemetry_merged.csv", low_memory=False)
    features_df = pd.read_csv(EXPORT_DIR / "race_2025_balanced_top5_lap_features.csv")

    telemetry_df["Brake"] = telemetry_df["Brake"].astype(str).str.lower().map({"true": 1.0, "false": 0.0})

    # Match the strongest handcrafted race baseline: no time features and no race-context features.
    base_drop = [
        "lap_key",
        "Driver",
        "season",
        "round",
        "event_name",
        "session_name",
        "LapNumber",
        "Team",
        "session_key",
        "TrackStatus",
        "lap_time_not_null",
        "dry_session",
        "accurate",
        "not_deleted",
        "not_fastf1_generated",
        "track_status_green",
        "not_pit_in_out",
        "within_2s_of_stint_median",
        "within_3s_of_stint_median",
        "throttle_min",
        "brake_min",
        "LapTimeSeconds",
        "stint_median_lap_seconds",
        "lap_time_delta_to_stint_median",
        "Compound",
        "TyreLife",
        "Stint",
        "SessionPart",
        "drs_active_frac",
    ]
    tab_features = [c for c in features_df.columns if c not in base_drop and not features_df[c].isna().all()]

    lap_rows = []
    sequence_arrays = []
    for lap_key, lap_df in telemetry_df.sort_values(["lap_key", "sample_idx"]).groupby("lap_key"):
        lap_meta = lap_df.iloc[0]
        channel_matrix = []
        for channel in SEQ_CHANNELS:
            series = pd.to_numeric(lap_df[channel], errors="coerce")
            channel_matrix.append(resample_sequence(series.values, SEQ_LEN))
        sequence_arrays.append(np.stack(channel_matrix, axis=-1))
        lap_rows.append(
            {
                "lap_key": lap_key,
                "Driver": lap_meta["Driver"],
                "session_key": lap_meta["session_key"],
                "event_name": lap_meta.get("event_name"),
                "round": int(lap_meta["round"]),
                "raw_len": len(lap_df),
            }
        )

    sequence_meta_df = pd.DataFrame(lap_rows)
    meta_df = sequence_meta_df.merge(features_df[["lap_key"] + tab_features], on="lap_key", how="inner")

    sequence_lookup = {row["lap_key"]: idx for idx, row in sequence_meta_df.reset_index().iterrows()}
    aligned_indices = meta_df["lap_key"].map(sequence_lookup).to_numpy()
    X_seq = np.stack(sequence_arrays)[aligned_indices].astype(np.float32)
    X_tab = meta_df[tab_features].astype(float).to_numpy(dtype=np.float32)
    y_labels = meta_df["Driver"].astype(str).to_numpy()
    groups = meta_df["session_key"].astype(str).to_numpy()

    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(y_labels)

    return {
        "tab_features": tab_features,
        "meta_df": meta_df,
        "X_seq": X_seq,
        "X_tab": X_tab,
        "y": y,
        "y_labels": y_labels,
        "groups": groups,
        "label_encoder": label_encoder,
    }


def run_models(data_bundle: dict, handcrafted_summary: pd.DataFrame):
    X_seq = data_bundle["X_seq"]
    X_tab = data_bundle["X_tab"]
    y = data_bundle["y"]
    groups = data_bundle["groups"]
    n_classes = len(np.unique(y))

    outer_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    outer_splits = []
    for fold_idx, (train_idx, test_idx) in enumerate(outer_cv.split(X_seq, y, groups), start=1):
        inner_cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE + fold_idx)
        train_inner_rel, val_rel = next(inner_cv.split(X_seq[train_idx], y[train_idx], groups[train_idx]))
        outer_splits.append(
            {
                "fold": fold_idx,
                "train_idx": train_idx,
                "train_inner_idx": train_idx[train_inner_rel],
                "val_idx": train_idx[val_rel],
                "test_idx": test_idx,
            }
        )

    split_rows = [
        {
            "fold": split["fold"],
            "n_train": len(split["train_idx"]),
            "n_train_inner": len(split["train_inner_idx"]),
            "n_val": len(split["val_idx"]),
            "n_test": len(split["test_idx"]),
        }
        for split in outer_splits
    ]

    model_defs = {
        "cnn": lambda: build_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
        "hybrid_cnn_tabular": lambda: build_hybrid_cnn_model(SEQ_LEN, len(SEQ_CHANNELS), X_tab.shape[1], n_classes),
        "gru": lambda: build_gru_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
        "lstm": lambda: build_lstm_model(SEQ_LEN, len(SEQ_CHANNELS), n_classes),
    }

    fold_rows = []
    history_rows = []
    summary_rows = []

    for model_name, builder in model_defs.items():
        oof_pred = np.full(len(y), -1, dtype=int)

        for split in outer_splits:
            train_idx = split["train_idx"]
            train_inner_idx = split["train_inner_idx"]
            val_idx = split["val_idx"]
            test_idx = split["test_idx"]

            seq_train_inner, seq_val, seq_test = standardize_sequence_by_train(
                X_seq[train_inner_idx], X_seq[val_idx], X_seq[test_idx]
            )
            seq_train_full, _, _ = standardize_sequence_by_train(
                X_seq[train_idx], X_seq[val_idx], X_seq[test_idx]
            )
            tab_train_inner, tab_val, tab_test = standardize_by_train(
                X_tab[train_inner_idx], X_tab[val_idx], X_tab[test_idx]
            )
            tab_train_full, _, _ = standardize_by_train(
                X_tab[train_idx], X_tab[val_idx], X_tab[test_idx]
            )

            y_train_inner = y[train_inner_idx]
            y_val = y[val_idx]
            y_train_full = y[train_idx]
            y_test = y[test_idx]

            callbacks = [
                tf.keras.callbacks.EarlyStopping(
                    monitor="val_loss",
                    patience=PATIENCE,
                    restore_best_weights=True,
                )
            ]

            tf.keras.backend.clear_session()
            model = builder()

            if model_name == "hybrid_cnn_tabular":
                history = model.fit(
                    [seq_train_inner, tab_train_inner],
                    y_train_inner,
                    validation_data=([seq_val, tab_val], y_val),
                    epochs=MAX_EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    callbacks=callbacks,
                )
                train_pred = np.argmax(model.predict([seq_train_full, tab_train_full], verbose=0), axis=1)
                test_pred = np.argmax(model.predict([seq_test, tab_test], verbose=0), axis=1)
            else:
                history = model.fit(
                    seq_train_inner,
                    y_train_inner,
                    validation_data=(seq_val, y_val),
                    epochs=MAX_EPOCHS,
                    batch_size=BATCH_SIZE,
                    verbose=0,
                    callbacks=callbacks,
                )
                train_pred = np.argmax(model.predict(seq_train_full, verbose=0), axis=1)
                test_pred = np.argmax(model.predict(seq_test, verbose=0), axis=1)

            oof_pred[test_idx] = test_pred

            history_rows.append(
                {
                    "model": model_name,
                    "fold": split["fold"],
                    "epochs_trained": len(history.history["loss"]),
                    "best_val_loss": float(np.min(history.history["val_loss"])),
                }
            )

            fold_rows.append(
                {
                    "model": model_name,
                    "fold": split["fold"],
                    "train_accuracy": accuracy_score(y_train_full, train_pred),
                    "test_accuracy": accuracy_score(y_test, test_pred),
                    "train_balanced_accuracy": balanced_accuracy_score(y_train_full, train_pred),
                    "test_balanced_accuracy": balanced_accuracy_score(y_test, test_pred),
                    "train_macro_f1": f1_score(y_train_full, train_pred, average="macro"),
                    "test_macro_f1": f1_score(y_test, test_pred, average="macro"),
                }
            )

        summary_rows.append(
            {
                "model": model_name,
                "n_samples": len(y),
                "n_drivers": n_classes,
                "n_sessions": len(np.unique(groups)),
                "oof_accuracy": accuracy_score(y, oof_pred),
                "oof_balanced_accuracy": balanced_accuracy_score(y, oof_pred),
                "oof_macro_f1": f1_score(y, oof_pred, average="macro"),
            }
        )

    fold_df = pd.DataFrame(fold_rows)
    grouped = (
        fold_df.groupby("model", dropna=False)
        .agg(
            mean_train_accuracy=("train_accuracy", "mean"),
            mean_test_accuracy=("test_accuracy", "mean"),
            mean_train_macro_f1=("train_macro_f1", "mean"),
            mean_test_macro_f1=("test_macro_f1", "mean"),
        )
        .reset_index()
    )
    grouped["mean_accuracy_gap"] = grouped["mean_train_accuracy"] - grouped["mean_test_accuracy"]
    grouped["mean_macro_f1_gap"] = grouped["mean_train_macro_f1"] - grouped["mean_test_macro_f1"]

    summary_df = pd.DataFrame(summary_rows).merge(grouped, on="model", how="left")

    handcrafted_best = handcrafted_summary.sort_values(
        ["oof_macro_f1", "oof_accuracy"], ascending=[False, False]
    ).head(1)
    compare_rows = []
    if not handcrafted_best.empty:
        base_acc = handcrafted_best["oof_accuracy"].iloc[0]
        base_f1 = handcrafted_best["oof_macro_f1"].iloc[0]
        compare_rows.append(
            {
                "model_family": "handcrafted_compact_logreg",
                "oof_accuracy": base_acc,
                "oof_macro_f1": base_f1,
                "accuracy_vs_handcrafted": 0.0,
                "macro_f1_vs_handcrafted": 0.0,
            }
        )
        for _, row in summary_df.iterrows():
            compare_rows.append(
                {
                    "model_family": row["model"],
                    "oof_accuracy": row["oof_accuracy"],
                    "oof_macro_f1": row["oof_macro_f1"],
                    "accuracy_vs_handcrafted": row["oof_accuracy"] - base_acc,
                    "macro_f1_vs_handcrafted": row["oof_macro_f1"] - base_f1,
                }
            )

    return (
        pd.DataFrame(split_rows),
        fold_df,
        pd.DataFrame(history_rows),
        summary_df.sort_values(["oof_macro_f1", "oof_accuracy"], ascending=[False, False]).reset_index(drop=True),
        pd.DataFrame(compare_rows),
    )


def main():
    handcrafted_summary = pd.read_csv(EXPORT_DIR / "race_2025_baseline_model_summary.csv")
    handcrafted_summary = handcrafted_summary[
        handcrafted_summary["model"] == "logistic_regression"
    ].copy()

    bundle = prepare_data()
    split_df, fold_df, history_df, summary_df, compare_df = run_models(bundle, handcrafted_summary)
    best_df = summary_df.head(1).reset_index(drop=True)

    export_csv(split_df, "race_2025_sequence_split_summary")
    export_csv(fold_df, "race_2025_sequence_fold_metrics")
    export_csv(history_df, "race_2025_sequence_training_history")
    export_csv(summary_df, "race_2025_sequence_model_summary")
    export_csv(best_df, "race_2025_sequence_best_models")
    export_csv(compare_df, "race_2025_sequence_vs_handcrafted")

    print("Race 2025 sequence modeling complete.")
    print(best_df.to_string(index=False))


if __name__ == "__main__":
    main()
